[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/04_neural_networks/04_neural_networks_solutions.ipynb)

# 04. Neural Networks — 연습 문제 해설

[04_neural_networks.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/04_neural_networks/04_neural_networks.ipynb) 끝의 연습 문제 4개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

> **읽는 법** — 먼저 직접 풀어본 뒤 보세요. 셀은 위에서부터 순서대로 실행해야 하고,
> 실행 결과는 저장되어 있지 않으니 직접 실행해야 출력이 나타납니다.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision matplotlib

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# GPU가 있으면 GPU, 없으면 CPU. 이후 .to(device)로 모델·데이터를 올린다
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)   # 시드 고정 — 가중치 초기값이 같아져 실험 조건이 통제된다
print("device:", device)

## 연습 1. XOR — 은닉 유닛 수를 8 -> 2로 줄이면?

In [ ]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]], device=device)   # tensor: GPU 연산과 자동 미분이 가능한 배열
Y = torch.tensor([[0.], [1.], [1.], [0.]], device=device)

def make_mlp(hidden_size):
    return nn.Sequential(
        nn.Linear(2, hidden_size),
        nn.Sigmoid(),
        nn.Linear(hidden_size, 1),
        nn.Sigmoid(),
    ).to(device)

def train(model, epochs=3000, lr=0.5):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)   # SGD: 기울기 반대 방향으로 lr만큼 가중치를 움직이는 최적화기
    loss_fn = nn.BCELoss()   # BCELoss: 이진 분류용 손실 함수
    losses = []
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(X), Y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

torch.manual_seed(0)
model_h2 = make_mlp(2)
losses_h2 = train(model_h2)

torch.manual_seed(0)
model_h8 = make_mlp(8)
losses_h8 = train(model_h8)

with torch.no_grad():
    print("hidden=2 예측:", model_h2(X).cpu().numpy().round(3).flatten(), " 최종 loss:", losses_h2[-1])
    print("hidden=8 예측:", model_h8(X).cpu().numpy().round(3).flatten(), " 최종 loss:", losses_h8[-1])

은닉 유닛 수만 다른 두 모델의 loss 곡선을 비교합니다.

In [ ]:
plt.plot(losses_h2, label="hidden=2")
plt.plot(losses_h8, label="hidden=8")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("은닉 유닛 수에 따른 XOR 학습")
plt.legend()
plt.show()

**해설**
- 이론적으로는 은닉 유닛 **2개**만 있어도 XOR을 표현할 수 있습니다 (XOR = 두 개의 직선 결정 경계 조합으로 구성 가능).
- 하지만 실제로 학습해보면 `hidden=2`는 초기 가중치(random seed)에 따라 **local minimum에 갇혀서 잘 못 풀 때가 많습니다** — 표현력은 충분해도 그 해를 SGD로 찾아내는 게 어렵습니다.
- `hidden=8`처럼 여유 있게 유닛을 두면(over-parameterize) 최적화 지형(loss landscape)이 더 우호적이 되어 안정적으로 수렴합니다.
- 실무 교훈: "이론적 최소 크기"보다 약간 여유 있는 모델이 오히려 학습이 더 쉬운 경우가 많습니다.

## 연습 2. 연쇄 법칙 — `w2`를 1.5에서 5.0으로 키우면?

본문 4절의 실습을 `w2` 값만 바꿔서 두 번 돌리고, 다섯 조각을 나란히 비교합니다.

In [ ]:
def chain_breakdown(w2_value):
    x = torch.tensor(1.0)
    y = torch.tensor(0.0)
    w1 = torch.tensor(0.8, requires_grad=True)
    w2 = torch.tensor(w2_value, requires_grad=True)

    h = torch.sigmoid(w1 * x)
    out = torch.sigmoid(w2 * h)
    loss = -(y * torch.log(out) + (1 - y) * torch.log(1 - out))
    loss.backward()

    o, hh = out.item(), h.item()
    return {
        "out": o,
        "dloss/dout": (o - y.item()) / (o * (1 - o)),
        "dout/dz2": o * (1 - o),
        "dz2/dh": w2.item(),
        "dh/dz1": hh * (1 - hh),
        "dz1/dw1": x.item(),
        "dw1": w1.grad.item(),
    }


a = chain_breakdown(1.5)
b = chain_breakdown(5.0)

print(f"{'':>12}{'w2=1.5':>12}{'w2=5.0':>12}")
print("-" * 36)
for key in ["out", "dloss/dout", "dout/dz2", "dz2/dh", "dh/dz1", "dz1/dw1", "dw1"]:
    print(f"{key:>12}{a[key]:>12.4f}{b[key]:>12.4f}")

**해설**

| 조각 | `w2=1.5` | `w2=5.0` | 바뀌었나 |
|---|---|---|---|
| `out` (예측값) | 0.7379 | 0.9692 | — |
| `dloss/dout` | 3.8150 | **32.4964** | 커짐 |
| `dout/dz2` | 0.1934 | **0.0298** | 작아짐 |
| `dz2/dh` | 1.5000 | **5.0000** | 커짐 (= `w2` 그 자체) |
| `dh/dz1` | 0.2139 | 0.2139 | **그대로** |
| `dz1/dw1` | 1.0000 | 1.0000 | **그대로** |
| **`dw1`** | 0.2368 | **1.0366** | 약 4.4배 |

**바뀐 것과 안 바뀐 것을 나누는 기준이 분명합니다.**

- **안 바뀐 두 조각(`dh/dz1`, `dz1/dw1`)은 `w2`보다 앞쪽**에 있습니다. `w2`는 `h`가 이미
  계산된 뒤에 등장하므로, `h`와 `z1`에 관련된 값은 `w2`가 무엇이든 그대로입니다.
- **`dz2/dh`는 정확히 `w2` 값 그 자체**입니다. `z2 = w2 × h`를 `h`로 미분하면 `w2`니까요.
  1.5 → 5.0으로 바꿨으니 그대로 1.5 → 5.0이 됩니다.

**흥미로운 것은 나머지 두 조각이 반대 방향으로 움직인다는 점입니다.**

`w2`를 키우면 `out`이 0.9692까지 올라갑니다. 정답이 0인데 0.97이라고 답한 것이니
**확신을 가지고 크게 틀린 상태**입니다. 그래서

- `dloss/dout`은 **폭증**합니다(3.8 → 32.5). 03번에서 배운 교차 엔트로피의 성질입니다 —
  확신에 찬 오답에 큰 벌점을 줍니다.
- 반대로 `dout/dz2`는 **급감**합니다(0.19 → 0.03). 시그모이드가 1에 가까운 **포화 구간**에
  들어가서 곡선이 거의 평평해졌기 때문입니다.

결과적으로 `dw1`은 0.2368 → 1.0366으로 커졌습니다. **곱셈의 연쇄에서는 조각 하나만 보고
결과를 예측할 수 없습니다.** 어떤 조각은 커지고 어떤 조각은 작아지며, 최종 기울기는 그
전부의 곱입니다. 본문 5절에서 시그모이드를 여러 층 쌓았을 때 기울기가 사라지는 것도
바로 이 곱셈이 한쪽으로만 몰린 경우입니다.

## 연습 3 & 4. MNIST — Dropout 유무 비교, 98%+ 달성하기

> **아래 셀은 세 부분입니다.**
> 1. `EPOCHS = 5` — 두 실험에 같은 조건을 씁니다
> 2. **Dropout 없음** 모델을 만들어 학습 (`torch.manual_seed(0)`으로 초기값 고정)
> 3. **Dropout 0.3** 모델을 같은 시드로 학습
>
> 시드를 매번 다시 고정하는 것이 핵심입니다. 그래야 **차이가 Dropout 때문**이라고 말할 수 있습니다.

In [ ]:
# ══════════════════════════════════════════════════════════════
# ① 데이터 준비 — MNIST를 내려받아 배치 단위로 꺼내 쓸 준비를 한다
# ══════════════════════════════════════════════════════════════
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.MNIST(root="../../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../../data", train=False, download=True, transform=transform)
# DataLoader: 데이터를 batch_size개씩 묶어 순회시켜 주는 도구.
# 학습용은 매 epoch 순서를 섞고(shuffle=True), 평가용은 섞지 않는다.
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

# ══════════════════════════════════════════════════════════════
# ② 모델 정의 — dropout_p 값에 따라 Dropout 층을 넣거나 뺀다
#    (이 문제의 핵심: 같은 구조에서 Dropout만 다르게 두고 비교하기 위함)
# ══════════════════════════════════════════════════════════════
class MnistMLP(nn.Module):
    def __init__(self, dropout_p=0.0):
        super().__init__()
        # Flatten: 28×28 이미지를 784개 숫자 한 줄로 편다 -> 256개 뉴런으로
        layers = [nn.Flatten(), nn.Linear(28 * 28, 256), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        layers += [nn.Linear(256, 128), nn.ReLU()]
        if dropout_p > 0:
            layers.append(nn.Dropout(dropout_p))
        layers.append(nn.Linear(128, 10))   # 마지막 층: 숫자 0~9에 대한 점수 10개
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# ══════════════════════════════════════════════════════════════
# ③ 평가 함수 — 맞힌 개수 / 전체 개수 = 정확도
# ══════════════════════════════════════════════════════════════
def evaluate(model, loader):
    model.eval()          # 평가 모드: Dropout을 끈다 (학습 때만 켜져야 하므로)
    correct, total = 0, 0
    with torch.no_grad():  # 기울기 계산을 끈다 — 평가에는 필요 없고 메모리만 먹는다
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)   # 10개 점수 중 가장 큰 것의 인덱스 = 예측 숫자
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total

# ══════════════════════════════════════════════════════════════
# ④ 학습 루프 — epoch마다 train/test 정확도를 기록해서 돌려준다
#    (두 값의 간격이 벌어지는 것이 곧 과적합)
# ══════════════════════════════════════════════════════════════
def train_mnist(model, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    train_accs, test_accs = [], []
    for epoch in range(epochs):
        model.train()      # 학습 모드: Dropout을 켠다
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            # PyTorch 학습 4단계 — 이 순서는 어떤 모델에서도 같다
            optimizer.zero_grad()             # 1. 이전 기울기 지우기
            loss = loss_fn(model(xb), yb)     # 2. 순전파 + 손실 계산
            loss.backward()                   # 3. 역전파 (기울기 계산)
            optimizer.step()                  # 4. 가중치 갱신
        train_acc = evaluate(model, train_loader)
        test_acc = evaluate(model, test_loader)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        print(f"  epoch {epoch+1}/{epochs}  train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")
    return train_accs, test_accs

Dropout 유무만 다르게 두고 각각 5 epoch씩 학습시킵니다.
**같은 시드에서 시작하므로 차이는 Dropout 때문**입니다.

In [ ]:
EPOCHS = 5

print("[Dropout 없음]")
torch.manual_seed(0)
model_no_dropout = MnistMLP(dropout_p=0.0).to(device)
train_accs_nd, test_accs_nd = train_mnist(model_no_dropout, EPOCHS)

print("\n[Dropout=0.3]")
torch.manual_seed(0)
model_dropout = MnistMLP(dropout_p=0.3).to(device)
train_accs_d, test_accs_d = train_mnist(model_dropout, EPOCHS)

train/test 정확도를 나란히 그립니다. **두 곡선의 간격이 과적합의 크기**입니다.

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs_range, train_accs_nd, "--", label="train (no dropout)")
axes[0].plot(epochs_range, test_accs_nd, label="test (no dropout)")
axes[0].set_title("Dropout 없음: train vs test")
axes[0].legend()

axes[1].plot(epochs_range, train_accs_d, "--", label="train (dropout=0.3)")
axes[1].plot(epochs_range, test_accs_d, label="test (dropout=0.3)")
axes[1].set_title("Dropout=0.3: train vs test")
axes[1].legend()
plt.show()

print(f"최종 gap (train-test) — Dropout 없음: {train_accs_nd[-1]-test_accs_nd[-1]:.4f}")
print(f"최종 gap (train-test) — Dropout=0.3 : {train_accs_d[-1]-test_accs_d[-1]:.4f}")

**해설 (연습 3)**
- Dropout 없이 학습하면 train accuracy가 test accuracy보다 눈에 띄게 높아지는 경향(=train-test gap 확대)이 나타납니다. 모델이 학습 데이터의 세부 패턴(노이즈 포함)까지 암기하기 시작했다는 신호입니다.
- Dropout=0.3을 적용하면 매 스텝마다 뉴런의 일부를 무작위로 꺼서 특정 뉴런 조합에 의존하지 못하게 하므로, train accuracy는 약간 낮아지지만 **train-test gap이 줄어듭니다** — 더 일반화가 잘 된다는 뜻입니다.
- MNIST는 비교적 쉬운 데이터셋이라 이 문제 규모에서는 차이가 크지 않을 수 있지만, 더 복잡한 데이터/모델일수록 Dropout의 효과가 뚜렷해집니다.

**해설 (연습 4)**
위 Dropout=0.3 모델을 5 epoch만 학습해도 보통 test accuracy가 97~98%대에 도달합니다. epoch을 8~10으로 늘리거나 은닉층을 하나 더 추가하면 98% 이상을 안정적으로 넘길 수 있습니다 (원본 강의 Lec 10의 목표와 동일). 정확도를 더 올리고 싶다면 CNN(다음 노트북)을 쓰는 것이 가장 효과적입니다.